# WTI Crude Oil Price Forecasting — Systematic Backtesting and Evaluation (Notebook 4)

This notebook simulates a rigorous production forecasting workflow:

1. Run a **rolling weekly backtest across 2025** using
   `energy_oil_backtest.yaml` for all candidate predictors.
2. Compute metrics — **CRPS** for 5/10/21-day trajectories.
3. Select the **top contender configurations** based solely on 2025
   historical performance (no peeking at 2026).
4. Let the contenders compete in the **2026 Protected Arena**
   (`energy_oil_eval.yaml`) during the geopolitical price shock —
   measuring adaptive real-time responsiveness and calibration.

All predictors use the same `Predictor` interface introduced in Notebooks 1–2.
Agent configs are imported from `energy_oil_forecasting.analyst_agent`.

---
## 1. Setup, Data Registration & Spec Loading

In [ ]:
import warnings
from pathlib import Path

import energy_oil_forecasting
import pandas as pd
from aieng.forecasting.evaluation import (
    MultiTargetBacktestSpec,
    cached_multi_backtest,
    describe_spec,
)
from energy_oil_forecasting.data import build_wti_service

warnings.filterwarnings("ignore")

data_service = build_wti_service()

spec_dir = Path(energy_oil_forecasting.__file__).parent / "specs"
with open(spec_dir / "energy_oil_backtest.yaml") as f:
    backtest_spec = MultiTargetBacktestSpec.model_validate_yaml(f.read())
with open(spec_dir / "energy_oil_eval.yaml") as f:
    eval_spec = MultiTargetBacktestSpec.model_validate_yaml(f.read())

print("━" * 72)
print("LOADED SPECIFICATIONS:")
print("━" * 72)
print(describe_spec(backtest_spec, data_service))
print(describe_spec(eval_spec, data_service))


---
## 2. Candidate Predictors

We reuse the shared wrappers from earlier notebooks — `ProphetPredictor` from
`prophet_baseline.py`, `ContinuousLLMPredictor` from `aieng`, and
`build_wti_agent_predictor()` for the news-grounded agent.

In [ ]:
from aieng.forecasting.methods import (
    ContinuousLLMPredictor,
    ContinuousLLMPredictorConfig,
    LastValuePredictor,
)
from energy_oil_forecasting.analyst_agent import (
    build_wti_agent_predictor,
    build_wti_news_config,
)
from energy_oil_forecasting.prophet_baseline import ProphetPredictor

lv = LastValuePredictor()
prophet = ProphetPredictor(predictor_id="prophet_daily")
llmp = ContinuousLLMPredictor(
    ContinuousLLMPredictorConfig(model="gemini/gemini-3.5-flash", n_samples=3)
)
news_agent = build_wti_agent_predictor(build_wti_news_config())

candidates = [lv, prophet, llmp, news_agent]
print("Candidate predictors:")
for c in candidates:
    print(f"  {c.predictor_id}")


---
## 3. Run the 2025 Historical Backtest

All 51 weekly origins in 2025 are evaluated for each predictor.
`cached_multi_backtest` caches results under `data/predictions/` so
subsequent runs are instant.

In [ ]:
print("Running 2025 rolling backtest (51 weekly origins × 4 predictors)...")
print("LLM/agent runs are expensive — first run will take several minutes.\n")

lv_results = cached_multi_backtest(lv, backtest_spec, data_service)
print("LastValue ✓")

prophet_results = cached_multi_backtest(prophet, backtest_spec, data_service)
print("Prophet ✓")

llmp_results = cached_multi_backtest(llmp, backtest_spec, data_service)
print("LLMP ✓")

news_results = cached_multi_backtest(news_agent, backtest_spec, data_service)
print("News agent ✓")

print("\nAll 2025 backtests complete.")

---
## 4. Compute Metrics and Select Contenders

We score each predictor on:
- **CRPS** (Continuous Ranked Probability Score) across the 5/10/21-day trajectory
- **MAE** at the 21-day horizon (point forecast accuracy)

The top 3 scorers (by mean CRPS) are selected as contenders for the
2026 protected arena. Selection is based solely on 2025 performance.

In [ ]:
from energy_oil_forecasting.analysis import score_backtest_results


all_results = [
    ("Naive (Last Value)", lv_results),
    ("Prophet", prophet_results),
    ("LLMP (Gemini 3.5-flash)", llmp_results),
    ("News-Grounded Agent", news_results),
]

leaderboard_rows = []
for name, results in all_results:
    scores = score_backtest_results(results, data_service)
    leaderboard_rows.append(
        {
            "Predictor": name,
            "Mean CRPS": scores.get("mean_crps", float("nan")),
            "MAE h=21d": scores.get("mae_h21", float("nan")),
        }
    )

df_leaderboard = pd.DataFrame(leaderboard_rows).set_index("Predictor")
df_leaderboard = df_leaderboard.sort_values("Mean CRPS")

print("━" * 72)
print("2025 HISTORICAL BACKTEST LEADERBOARD:")
print("━" * 72)
print(df_leaderboard.to_string())

top3 = df_leaderboard.head(3).index.tolist()
print(f"\nSelected contenders for 2026 arena: {top3}")

---
## 5. The 2026 Protected Arena Competition

We evaluate the selected contenders on **8 weekly origins in early 2026**
(`energy_oil_eval.yaml`) — a period of major geopolitical volatility as
Persian Gulf shipping-lane closures drove WTI from ~$71 to above $100.

This is a **prospective evaluation**: the 2026 data was not seen during
contender selection. News-grounded agents retrieve information with a strict
temporal cutoff at each origin, approximating a genuine live-test environment.

In [ ]:
# Map selected contender names back to predictor objects
contender_map = {
    "Naive (Last Value)": lv,
    "Prophet": prophet,
    "LLMP (Gemini 3.5-flash)": llmp,
    "News-Grounded Agent": news_agent,
}

print("Running 2026 protected arena evaluation...")
eval_results = {}
for name in top3:
    predictor = contender_map[name]
    eval_results[name] = cached_multi_backtest(predictor, eval_spec, data_service)
    print(f"  {name} ✓")

print("\n2026 evaluation complete.")

---
## 6. Visualisation & Scorecard

We compare how each contender reacted as the price shock unfolded.
Statistical models like Prophet expect mean-reversion and miss the breakout.
The news-grounded agent reads real-time intelligence and adjusts its forecast
accordingly — at the cost of higher compute and latency.

In [ ]:
from energy_oil_forecasting.analysis import score_backtest_results


scorecard_rows = []
for name in top3:
    scores = score_backtest_results(eval_results[name], data_service)
    scorecard_rows.append(
        {
            "Predictor": name,
            "Mean CRPS (2026)": scores.get("mean_crps", float("nan")),
            "MAE h=21d (2026)": scores.get("mae_h21", float("nan")),
            "80% CI Coverage": scores.get("coverage_80", float("nan")),
        }
    )

df_scorecard = pd.DataFrame(scorecard_rows).set_index("Predictor")
df_scorecard = df_scorecard.sort_values("Mean CRPS (2026)")

print("━" * 72)
print("FINAL 2026 PROTECTED ARENA SCORECARD:")
print("━" * 72)
print(df_scorecard.to_string())

---
## 7. Core Takeaways

1. **Statistical models** (Prophet, Last Value) are strong in stable regimes.
   During structural price shocks they extrapolate past trends, missing the
   breakout and producing catastrophically narrow intervals.

2. **Direct-prompt LLMPs** have an implicit knowledge cutoff. For 2026
   origins they may have partial training signal about early 2026 events,
   but cannot access post-cutoff news in real time.

3. **News-grounded agents** with bounded search incorporate real-time
   market intelligence, enabling a much faster response to structural
   shocks — at higher compute cost and non-zero leakage risk through
   the search tool.

4. **The `Predictor` abstraction makes all of this composable.** The same
   backtest harness, scoring functions, and visualisation tools work
   equally for Prophet, LLMP, and agent predictors.